# 06 · Textos: noticias y filings

**Objetivo:** reunir y limpiar las dos fuentes de **texto no estructurado** del proyecto: las noticias de cada empresa y el texto de sus presentaciones a la SEC (filings). Aquí no se calcula ningún sentimiento: solo se prepara el texto que usarán los modelos de los notebooks 07 y 08.

**Entradas:**
- `data/raw/fnspid/All_external.csv`: dataset FNSPID (se descarga en la sección 1 si no existe)
- `data/cache/universo_recortado.parquet` (notebook 02): las 1.000 empresas
- `data/cache/senales.parquet` (notebook 05): para saber qué empresas tienen señal
- Clave `NEWSAPI_KEY` en el `.env`

**Salidas:**
- `data/cache/noticias_fnspid.parquet`: titulares de nuestras empresas, oct 2016 – jun 2020
- `data/cache/noticias_newsapi.parquet`: noticias del último mes de las empresas con señal, se acumulan en cada ejecución
- *(próxima sección: texto de los filings)*

In [11]:
import re

import pandas as pd
import plotly.express as px
import requests
from dotenv import load_dotenv

from funciones import CARPETA_CACHE

load_dotenv();

## 1. ¿Qué noticias históricas tenemos de nuestras empresas?

Las APIs gratuitas de noticias solo dan las últimas semanas. Para tener histórico se usa **FNSPID** (Dong, Fan y Peng, 2024, [arXiv 2402.06698](https://arxiv.org/abs/2402.06698)), un dataset público de noticias financieras en el que cada titular lleva el **ticker** de la empresa. Licencia CC BY-NC 4.0: uso académico citando el artículo.

### 1.1 Descarga

El fichero ocupa 5,7 GB. Solo se descarga si no está ya en disco.

In [12]:
URL_FNSPID = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/All_external.csv"
RUTA_FNSPID = CARPETA_CACHE.parent / "raw" / "fnspid" / "All_external.csv"
TAMANO_FNSPID = 5_731_397_037   # bytes, comprobado con el servidor el 19 oct 2026

if not RUTA_FNSPID.exists():
    print("Descargando FNSPID (5,7 GB)...")
    RUTA_FNSPID.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(URL_FNSPID, stream=True, timeout=60) as respuesta:
        respuesta.raise_for_status()
        with open(RUTA_FNSPID, "wb") as fichero:
            for bloque in respuesta.iter_content(chunk_size=8 * 1024 * 1024):
                fichero.write(bloque)

tamano = RUTA_FNSPID.stat().st_size
if tamano != TAMANO_FNSPID:
    raise RuntimeError(f"FNSPID incompleto: {tamano} bytes en disco, se esperaban {TAMANO_FNSPID}. Borra el fichero y vuelve a ejecutar.")
print(f"FNSPID en disco: {tamano / 1e9:.2f} GB")

FNSPID en disco: 5.73 GB


### 1.2 Cómo es el fichero por dentro

Antes de procesarlo, las primeras filas tal como vienen:

In [13]:
pd.read_csv(RUTA_FNSPID, nrows=5)

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,2020-06-05 06:30:54 UTC,Stocks That Hit 52-Week Highs On Friday,A,https://www.benzinga.com/news/20/06/16190091/stocks-that-hit-52-week-highs-on-friday,Benzinga Insights,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-06-03 06:45:20 UTC,Stocks That Hit 52-Week Highs On Wednesday,A,https://www.benzinga.com/news/20/06/16170189/stocks-that-hit-52-week-highs-on-wednesday,Benzinga Insights,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-05-26 00:30:07 UTC,71 Biggest Movers From Friday,A,https://www.benzinga.com/news/20/05/16103463/71-biggest-movers-from-friday,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-05-22 08:45:06 UTC,46 Stocks Moving In Friday's Mid-Day Session,A,https://www.benzinga.com/news/20/05/16095921/46-stocks-moving-in-fridays-mid-day-session,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-05-22 07:38:59 UTC,"B of A Securities Maintains Neutral on Agilent Technologies, Raises Price Target to $88",A,https://www.benzinga.com/news/20/05/16095304/b-of-a-securities-maintains-neutral-on-agilent-technologies-r...,Vick Meyer,NaN,NaN,NaN,NaN,NaN,NaN


Nos interesan cuatro columnas: `Date` (fecha), `Article_title` (titular), `Stock_symbol` (ticker) y `Publisher` (medio). La columna `Article` (texto completo) está vacía en casi todas las filas de nuestras empresas, así que no se usa.

### 1.3 Quedarse con nuestras empresas desde octubre de 2016

Con 13 millones de filas el fichero no cabe cómodo en memoria, así que se lee **por trozos** de 500.000 filas y de cada trozo se guardan solo las filas de nuestras 1.000 empresas desde octubre de 2016 (el inicio de nuestros precios).

FNSPID escribe las clases de acciones con punto (`BRK.B`) y nosotros con guion (`BRK-B`): se unifica antes de filtrar. El resultado se guarda en caché para no volver a leer los 5,7 GB.

In [14]:
RUTA_NOTICIAS_FNSPID = CARPETA_CACHE / "noticias_fnspid.parquet"
universo = pd.read_parquet(CARPETA_CACHE / "universo_recortado.parquet")
nuestros_tickers = set(universo["ticker"])

if RUTA_NOTICIAS_FNSPID.exists():
    filtrado = None
    print("Ya filtrado en una ejecución anterior: se lee de la caché en la sección 1.5")
else:
    trozos = []
    columnas = ["Date", "Article_title", "Stock_symbol", "Publisher"]
    for trozo in pd.read_csv(RUTA_FNSPID, usecols=columnas, dtype=str, chunksize=500_000):
        trozo["ticker"] = trozo["Stock_symbol"].str.replace(".", "-", regex=False)
        trozo["fecha"] = pd.to_datetime(trozo["Date"].str[:10])
        es_nuestro = trozo["ticker"].isin(nuestros_tickers) & (trozo["fecha"] >= "2016-10-01")
        trozos.append(trozo[es_nuestro])
    filtrado = pd.concat(trozos, ignore_index=True)
    filtrado = filtrado.rename(columns={"Article_title": "titular", "Publisher": "medio"})[["fecha", "ticker", "titular", "medio"]]
    print(f"Titulares de nuestras empresas desde oct 2016: {len(filtrado):,}")

Ya filtrado en una ejecución anterior: se lee de la caché en la sección 1.5


### 1.4 ¿Cuánto ruido traen?

Al mirar los titulares aparecen dos problemas:

1. **Duplicados:** el mismo titular repetido para la misma empresa.
2. **Titulares que no hablan de la empresa:** FNSPID asigna a cada empresa todo lo que aparece en su página de noticias, y muchos titulares son de otras empresas o del mercado en general. Ejemplo real: *"Altria: Income And Growth Together In A Defensive Business"* asignado a Texas Instruments.

Para el segundo se marca si el titular **menciona a la empresa**: su ticker como palabra suelta, o la palabra clave de su nombre. La palabra clave es la primera palabra del nombre (`HORMEL FOODS CORP` → `HORMEL`), o las dos primeras si la primera es demasiado común (`AMERICAN AIRLINES`).

In [15]:
PALABRAS_DEMASIADO_COMUNES = {
    "AMERICAN", "GENERAL", "FIRST", "UNITED", "NATIONAL", "INTERNATIONAL", "TEXAS", "WESTERN", "EASTERN",
    "SOUTHERN", "NORTHERN", "GLOBAL", "THE", "NEW", "CENTRAL", "PACIFIC", "ATLANTIC", "ADVANCED", "APPLIED",
    "DIGITAL", "PUBLIC", "ROYAL", "STANDARD", "UNIVERSAL", "CAPITAL", "BANK", "ENERGY", "HEALTH", "CHINA",
    "CANADIAN", "PACKAGING", "ALLIED", "CONSOLIDATED", "WASTE",
}


def palabra_clave_empresa(nombre: str) -> str:
    """Primera palabra del nombre en mayúsculas, o las dos primeras si la primera es demasiado común."""
    palabras = re.sub(r"[^A-Za-z0-9& ]", " ", nombre).upper().split()
    if palabras[0] in PALABRAS_DEMASIADO_COMUNES and len(palabras) > 1:
        return palabras[0] + " " + palabras[1]
    return palabras[0]


def menciona_empresa(titular: str, ticker: str, palabra_clave: str) -> bool:
    """True si el titular contiene el ticker o la palabra clave como palabras sueltas."""
    titular = titular.upper()
    if re.search(r"\b" + re.escape(ticker) + r"\b", titular):
        return True
    # Una palabra clave de 1-2 letras (p. ej. "X") aparecería en cualquier titular: solo vale el ticker
    return len(palabra_clave) >= 3 and re.search(r"\b" + re.escape(palabra_clave) + r"\b", titular) is not None


universo["palabra_clave"] = universo["name"].map(palabra_clave_empresa)
universo[["ticker", "name", "palabra_clave"]].sample(8, random_state=1)

,ticker,name,palabra_clave
507,LH,LABCORP HOLDINGS INC.,LABCORP
818,FUTU,Futu Holdings Ltd,FUTU
452,QXO,"QXO, Inc.",QXO
368,CRS,CARPENTER TECHNOLOGY CORP,CARPENTER
242,TDG,TransDigm Group INC,TRANSDIGM
929,MTN,VAIL RESORTS INC,VAIL
262,EOG,EOG RESOURCES INC,EOG
810,BEN,FRANKLIN TEMPLETON INC,FRANKLIN


### 1.5 Limpiar y guardar

Se quitan los duplicados (misma empresa y mismo titular) y se añade la columna `sobre_la_empresa`. **No se borran** los titulares que no hablan de la empresa: no sirven para el sentimiento de esa empresa, pero sí como texto financiero para preentrenar los modelos del notebook 07.

In [16]:
if filtrado is not None:
    noticias = filtrado.dropna(subset=["titular"]).drop_duplicates(subset=["ticker", "titular"])
    noticias = noticias.merge(universo[["ticker", "palabra_clave"]], on="ticker")
    noticias["sobre_la_empresa"] = [
        menciona_empresa(titular, ticker, palabra)
        for titular, ticker, palabra in zip(noticias["titular"], noticias["ticker"], noticias["palabra_clave"])
    ]
    noticias = noticias.drop(columns="palabra_clave").sort_values(["ticker", "fecha"]).reset_index(drop=True)
    noticias.to_parquet(RUTA_NOTICIAS_FNSPID, index=False)
    print(f"Duplicados eliminados: {len(filtrado) - len(noticias):,}")

noticias = pd.read_parquet(RUTA_NOTICIAS_FNSPID)
print(f"Titulares guardados: {len(noticias):,} · sobre la propia empresa: {noticias['sobre_la_empresa'].mean():.0%}")

Titulares guardados: 464,983 · sobre la propia empresa: 42%


Ejemplos de cada tipo, para comprobar a ojo que la marca tiene sentido:

In [17]:
pd.set_option("display.max_colwidth", 110)
display(noticias[noticias["sobre_la_empresa"]].sample(6, random_state=4)[["fecha", "ticker", "titular"]])
display(noticias[~noticias["sobre_la_empresa"]].sample(6, random_state=4)[["fecha", "ticker", "titular"]])

,fecha,ticker,titular
245912,2016-11-22,KKR,KKR launches $4.5B offer for Calsonic Kansei
199301,2017-03-06,HAL,Halliburton Announces CFO Mark McCollum Leaving Co.; General Counsel Robb Voyles Named Interim CFO
401693,2019-10-24,SYK,What's in the Offing for Stryker's (SYK) Earnings in Q3?
368479,2019-05-13,RGLD,"RBC Capital Downgrades Royal Gold, Inc. - Common Stock to Sector Perform"
412118,2020-05-15,TMO,"Blue Pool Management Ltd. Buys Adobe Inc, Facebook Inc, Thermo Fisher Scientific Inc, Sells ..."
218076,2020-03-17,ICLR,"SunTrust Robinson Humphrey Maintains Hold on Icon, Lowers Price Target to $131"


,fecha,ticker,titular
364904,2017-08-01,REGN,Stocks Rally: 2 Leaders Break Out On Earnings
172425,2019-12-14,FE,Stocks To Watch: Investors Dig Into Trade War Halt
156143,2019-08-23,EOG,Shares of several oil & gas companies are trading lower on escalated trade tensions after China announced ...
116267,2018-01-15,CSX,Sales Growth + Dividend Growth = Danger!
399521,2017-01-04,SUI,Can You Afford To Retire? Current And Future Portfolio Income
397867,2019-01-07,STZ,New Age Beverages: Morinda Acquisition To Lead To Higher Revenues


### 1.6 Cobertura: qué empresas y qué años

Solo cuentan los titulares sobre la propia empresa, que son los que se usarán para su sentimiento.

In [18]:
sobre_empresa = noticias[noticias["sobre_la_empresa"]]
por_empresa = sobre_empresa.groupby("ticker").size()

print(f"Periodo: {noticias['fecha'].min().date()} → {noticias['fecha'].max().date()}")
print(f"Empresas con algún titular propio: {len(por_empresa)} de {len(universo)}")
print(f"Titulares propios por empresa: mediana {por_empresa.median():.0f} · 10 % de las empresas con menos de {por_empresa.quantile(0.1):.0f}")

universo["con_noticias"] = universo["ticker"].isin(por_empresa.index)
print(universo.groupby("exchange")["con_noticias"].mean().mul(100).round(0).rename("% de empresas con noticias").to_string())

Periodo: 2016-10-01 → 2020-06-11
Empresas con algún titular propio: 633 de 1000
Titulares propios por empresa: mediana 265 · 10 % de las empresas con menos de 75
exchange
NYSE      69.0
Nasdaq    54.0


In [19]:
por_mes = sobre_empresa.groupby(pd.Grouper(key="fecha", freq="ME")).size().reset_index(name="titulares")
grafico = px.bar(por_mes, x="fecha", y="titulares",
                 title="Titulares sobre la propia empresa, por mes (FNSPID, nuestras 1.000 empresas)")
grafico.show(renderer="notebook_connected")

**Conclusión:** FNSPID da noticias de unas dos terceras partes de nuestras empresas entre octubre de 2016 y junio de 2020. Menos de la mitad de los titulares tratan de la propia empresa; esos son los que medirán su sentimiento, y el resto se aprovecha como corpus. De las 367 empresas sin noticias, 119 empezaron a cotizar después de junio de 2020; las demás (incluidas algunas grandes, como Cardinal Health o Ferrari) simplemente no están en FNSPID. Ninguna de ellas podrá participar en la comparación noticias/filings, y esa limitación se documenta.

## 2. ¿Qué se dice ahora de las empresas con señal?

FNSPID termina en junio de 2020. Para el screener del día y para el test que se etiqueta a mano hacen falta noticias **recientes**. Se usa **NewsAPI**, con los límites del plan gratuito medidos el 19 oct 2026:

| Límite | Valor |
|---|---|
| Peticiones | 100 al día |
| Antigüedad máxima | un mes |
| Resultados por consulta | 100 como máximo |
| Retraso | ~1 día |

Con 100 peticiones no se pueden pedir las 1.000 empresas cada día, así que se piden solo las que tienen **señal el último día** de nuestros precios, en grupos de 3 por consulta.

Dos decisiones tomadas al ver los datos reales:

- **Solo medios financieros** (Yahoo Finance, MarketWatch, Benzinga, CNBC, PR Newswire…). Sin este filtro, buscar "Garmin" devuelve sorteos de relojes y paquetes de software.
- **Solo en el titular** (`searchIn=title`), igual que la marca `sobre_la_empresa` de FNSPID.

In [20]:
import os

URL_NEWSAPI = "https://newsapi.org/v2/everything"
MEDIOS_FINANCIEROS = [
    "finance.yahoo.com", "marketwatch.com", "fool.com", "seekingalpha.com", "benzinga.com", "cnbc.com",
    "investors.com", "zacks.com", "barrons.com", "wsj.com", "bloomberg.com", "reuters.com", "forbes.com",
    "businesswire.com", "prnewswire.com", "globenewswire.com", "biztoc.com", "thestreet.com", "investing.com",
    "nasdaq.com", "businessinsider.com", "ft.com", "axios.com", "gurufocus.com", "insidermonkey.com", "247wallst.com",
]


def buscar_titulares_newsapi(palabras: list[str], desde: pd.Timestamp) -> list[dict]:
    """Busca en los titulares de medios financieros cualquiera de las palabras, desde la fecha dada.

    Devuelve la lista de artículos tal como la da NewsAPI. Lanza RuntimeError si falta la
    clave o si la API responde con un error (por ejemplo, el límite diario agotado).
    """
    clave = os.getenv("NEWSAPI_KEY", "").strip()
    if not clave:
        raise RuntimeError("Falta NEWSAPI_KEY en el .env")
    parametros = {
        "q": " OR ".join(f'"{palabra}"' for palabra in palabras),
        "searchIn": "title",
        "domains": ",".join(MEDIOS_FINANCIEROS),
        "language": "en",
        "from": desde.strftime("%Y-%m-%d"),
        "sortBy": "publishedAt",
        "pageSize": 100,
    }
    respuesta = requests.get(URL_NEWSAPI, params=parametros, headers={"X-Api-Key": clave}, timeout=30)
    datos = respuesta.json()
    if respuesta.status_code != 200:
        raise RuntimeError(f"NewsAPI respondió {respuesta.status_code}: {datos.get('code')} - {datos.get('message')}")
    return datos["articles"]

Así llega una respuesta real (una consulta de prueba con Garmin): campos y un ejemplo. Se guarda en caché para no gastar una petición en cada ejecución.

In [21]:
import json

RUTA_PRUEBA_NEWSAPI = CARPETA_CACHE / "newsapi_respuesta_ejemplo.json"
if RUTA_PRUEBA_NEWSAPI.exists():
    prueba = json.loads(RUTA_PRUEBA_NEWSAPI.read_text())
else:
    prueba = buscar_titulares_newsapi(["Garmin"], desde=pd.Timestamp.today() - pd.Timedelta(days=29))
    RUTA_PRUEBA_NEWSAPI.write_text(json.dumps(prueba, indent=1))
print(f"Artículos: {len(prueba)} · campos: {list(prueba[0])}")
prueba[0]

Artículos: 9 · campos: ['source', 'author', 'title', 'description', 'url', 'urlToImage', 'publishedAt', 'content']


{'source': {'id': None, 'name': 'PRNewswire'},
 'author': None,
 'title': 'Garmin selects Ventura, California as newest Garmin Marathon Series location',
 'description': 'Event will showcase breathtaking mountain and ocean scenery in August 2027 OLATHE, Kan., Oct. 14, 2026 /PRNewswire/ -- Garmin (NYSE: GRMN) today announced the Garmin Marathon Series will expand to a fourth location in Ventura, California, on Sunday, Aug. 22, …',
 'url': 'https://www.prnewswire.com/news-releases/garmin-selects-ventura-california-as-newest-garmin-marathon-series-location-302908287.html',
 'urlToImage': 'https://mmx.prnewswire.com/media/MS1128164/Garmin-Marathon-Series.jpg?id=OA3016159&p=facebook',
 'publishedAt': '2026-10-14T14:02:00Z',
 'content': 'Event will showcase breathtaking mountain and ocean scenery in August 2027\r\nOLATHE, Kan., Oct. 14, 2026 /PRNewswire/ -- Garmin (NYSE: GRMN) today announced the Garmin Marathon Series will expand to a… [+5337 chars]'}

### 2.1 Descargar y acumular

Las empresas a consultar son las que tienen señal el último día de `senales.parquet`. Cada titular se asigna a las empresas de su grupo que **menciona** (la misma regla que en FNSPID).

Como NewsAPI solo guarda un mes, cada ejecución **se suma** a lo ya guardado (sin repetir artículos). Si ya se descargó hoy, no se vuelve a pedir nada: así re-ejecutar el notebook no gasta el cupo diario.

In [22]:
RUTA_NOTICIAS_NEWSAPI = CARPETA_CACHE / "noticias_newsapi.parquet"
EMPRESAS_POR_CONSULTA = 3
hoy = pd.Timestamp.today().normalize()

senales = pd.read_parquet(CARPETA_CACHE / "senales.parquet", columns=["fecha", "ticker", "senal"])
ultimo_dia = senales["fecha"].max()
con_senal = senales[(senales["fecha"] == ultimo_dia) & senales["senal"].isin(["compra", "venta"])]
empresas = universo[universo["ticker"].isin(con_senal["ticker"]) & (universo["palabra_clave"].str.len() >= 3)]
print(f"Empresas con señal el {ultimo_dia.date()}: {len(empresas)} → {-(-len(empresas) // EMPRESAS_POR_CONSULTA)} consultas")

guardadas = pd.read_parquet(RUTA_NOTICIAS_NEWSAPI) if RUTA_NOTICIAS_NEWSAPI.exists() else None

if guardadas is not None and guardadas["descargado"].max() >= hoy:
    print("Ya se descargó hoy: no se hace ninguna petición")
else:
    filas = []
    consultas_llenas = 0
    for inicio in range(0, len(empresas), EMPRESAS_POR_CONSULTA):
        grupo = empresas.iloc[inicio : inicio + EMPRESAS_POR_CONSULTA]
        articulos = buscar_titulares_newsapi(list(grupo["palabra_clave"]), desde=hoy - pd.Timedelta(days=29))
        if len(articulos) == 100:
            consultas_llenas += 1
        for articulo in articulos:
            for empresa in grupo.itertuples():
                if menciona_empresa(articulo["title"], empresa.ticker, empresa.palabra_clave):
                    filas.append({
                        "fecha": pd.Timestamp(articulo["publishedAt"][:10]),
                        "ticker": empresa.ticker,
                        "titular": articulo["title"],
                        "resumen": articulo["description"],
                        "medio": articulo["source"]["name"],
                        "url": articulo["url"],
                        "descargado": hoy,
                    })
    nuevas = pd.DataFrame(filas)
    print(f"Artículos nuevos asignados: {len(nuevas)} · consultas que llegaron al tope de 100: {consultas_llenas}")
    if guardadas is not None:
        nuevas = pd.concat([guardadas, nuevas], ignore_index=True)
    guardadas = nuevas.drop_duplicates(subset=["ticker", "url"])
    guardadas.to_parquet(RUTA_NOTICIAS_NEWSAPI, index=False)

noticias_recientes = pd.read_parquet(RUTA_NOTICIAS_NEWSAPI)
print(f"Noticias recientes guardadas: {len(noticias_recientes)} · empresas: {noticias_recientes['ticker'].nunique()}"
      f" · de {noticias_recientes['fecha'].min().date()} a {noticias_recientes['fecha'].max().date()}")

Empresas con señal el 2026-10-09: 99 → 33 consultas
Ya se descargó hoy: no se hace ninguna petición
Noticias recientes guardadas: 1459 · empresas: 89 · de 2026-09-20 a 2026-10-18


In [23]:
display(noticias_recientes.sample(8, random_state=1)[["fecha", "ticker", "medio", "titular"]])
noticias_recientes["medio"].value_counts().head(8)

,fecha,ticker,medio,titular
1108,2026-10-02,LYFT,Biztoc.com,"Lyft opened an 80,000-square-foot facility in Tennessee. It isn't a data center"
111,2026-10-16,MRNA,Biztoc.com,Moderna drops 11% amid neutral rating
1273,2026-10-09,PNW,PRNewswire,The Inner Circle acknowledges Dr. Reuben Azad as a Pinnacle Lifetime Member Inner Circle of Excellence
1147,2026-10-13,GME,Biztoc.com,"Take-Two Interactive Slides 5% as Wells Fargo Trims Estimates Behind $300 Target; Roblox Holds Flat, GameS..."
368,2026-09-21,SHOP,Biztoc.com,Shopify to Use Meta’s Muse for Agentic Checkout
37,2026-10-15,PLTR,Biztoc.com,ServiceNow vs. Palantir Technologies: Which Technology Stock Is a Better Buy in 2026?
1128,2026-10-12,WING,PRNewswire,Meauxmentum Strategies & Investments Celebrates Opening of 50th Wingstop Restaurant in North Little Rock
994,2026-09-22,SNAP,24/7 Wall St.,"Pinterest Sinks 5% as Month-Long Slide Deepens; Reddit Slips, Snap Stays Put"


medio
Biztoc.com                   1103
PRNewswire                    281
24/7 Wall St.                  24
CNBC                           23
Business Insider               19
Investor's Business Daily       6
TheStreet                       2
Forbes                          1
Name: count, dtype: int64

**Conclusión:** NewsAPI aporta titulares **y resumen** de las empresas con señal, solo del último mes. Casi todo llega a través de Biztoc (un agregador) y PR Newswire (notas de prensa de las propias empresas). La regla de mención falla a veces con nombres que son palabras comunes: a Pinnacle West (`PNW`) se le asigna un titular sobre un "Pinnacle Lifetime Member". Algunas consultas llegan al tope de 100 artículos, así que de las empresas más populares puede faltar alguna noticia. Cada ejecución en un día distinto suma noticias, así que el histórico reciente crece mientras se trabaja en el proyecto.

## Comprobaciones

In [24]:
# Solo nuestras empresas y solo el periodo pedido
assert noticias["ticker"].isin(nuestros_tickers).all()
assert noticias["fecha"].min() >= pd.Timestamp("2016-10-01")

# Sin titulares vacíos ni duplicados por empresa
assert noticias["titular"].notna().all()
assert not noticias.duplicated(subset=["ticker", "titular"]).any()

# La regla de mención, con casos escritos a mano
assert palabra_clave_empresa("HORMEL FOODS CORP /DE/") == "HORMEL"
assert palabra_clave_empresa("AMERICAN AIRLINES GROUP INC.") == "AMERICAN AIRLINES"
assert menciona_empresa("FedEx (FDX) Stock Pops On Q3 Beat", "FDX", "FEDEX")
assert menciona_empresa("Garmin Reports EASA Approval", "GRMN", "GARMIN")
assert not menciona_empresa("Altria: Income And Growth Together", "TXN", "TEXAS INSTRUMENTS")
assert not menciona_empresa("Next steps for X-ray makers", "XE", "X")   # palabra clave demasiado corta
print("Noticias FNSPID: OK")

Noticias FNSPID: OK


In [25]:
# Noticias recientes: sin duplicados, todas mencionan a su empresa y dentro del mes que permite NewsAPI
assert not noticias_recientes.duplicated(subset=["ticker", "url"]).any()
claves = universo.set_index("ticker")["palabra_clave"]
assert all(menciona_empresa(t, k, claves[k]) for t, k in zip(noticias_recientes["titular"], noticias_recientes["ticker"]))
assert (noticias_recientes["descargado"] - noticias_recientes["fecha"] <= pd.Timedelta(days=31)).all()
print("Noticias NewsAPI: OK")

Noticias NewsAPI: OK
